In [4]:
import cv2
import pandas as pd
import numpy as np
import os
from ultralytics import YOLO

# Path where pose samples will be saved
CSV_PATH = 'data/posture_data.csv'

In [5]:
# Load YOLOv8 pose model (downloads automatically on first run)
model = YOLO('yolov8n-pose.pt')

# COCO keypoint names for reference
KEYPOINT_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]

# Build column names: kp_0_x, kp_0_y, kp_0_conf, ..., kp_16_conf
# 17 keypoints x 3 values (x, y, confidence) = 51 features
columns = []
for i in range(17):
    columns += [f'kp_{i}_x', f'kp_{i}_y', f'kp_{i}_conf']
columns += ['label']

print(f'Total features per frame: {len(columns) - 1}')  # 51
print(f'Keypoints: {KEYPOINT_NAMES}')

Total features per frame: 51
Keypoints: ['nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear', 'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle']


In [ ]:
collected_rows = []   # holds all recorded samples for this session

COUNTDOWN_SECS = 3    # seconds to get into position
FRAMES_TO_CAPTURE = 30  # frames recorded per keypress

cap = cv2.VideoCapture(0)
print("Webcam open.")
print("Press 'g' = start good posture capture | 's' = start slouch capture | 'q' = quit")

def run_countdown(cap, model, label, seconds, n_frames):
    """Counts down on screen, then captures n_frames and returns them."""
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    rows = []

    # --- Countdown phase ---
    for i in range(seconds, 0, -1):
        deadline = cv2.getTickCount() + cv2.getTickFrequency()  # 1 second
        while cv2.getTickCount() < deadline:
            ret, frame = cap.read()
            if not ret:
                break
            results = model(frame, verbose=False)
            annotated = results[0].plot() if results[0].keypoints is not None else frame.copy()
            label_text = "GOOD POSTURE" if label == 0 else "SLOUCH"
            cv2.putText(annotated, f"Get into position: {label_text}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.putText(annotated, f"Starting in {i}...", (10, 70),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
            cv2.imshow('Collect Posture Data', annotated)
            cv2.waitKey(1)

    # --- Capture phase ---
    captured = 0
    while captured < n_frames:
        ret, frame = cap.read()
        if not ret:
            break
        results = model(frame, verbose=False)
        annotated = results[0].plot() if results[0].keypoints is not None else frame.copy()

        if results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
            kps = results[0].keypoints.data[0].cpu().numpy()
            rows.append(kps.flatten().tolist() + [label])
            captured += 1

        label_text = "GOOD POSTURE" if label == 0 else "SLOUCH"
        cv2.putText(annotated, f"Recording {label_text}...", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(annotated, f"{captured}/{n_frames} frames", (10, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
        cv2.imshow('Collect Posture Data', annotated)
        cv2.waitKey(1)

    return rows


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)
    annotated = results[0].plot() if results[0].keypoints is not None else frame.copy()

    cv2.putText(annotated, "g=good  s=slouch  q=quit", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(annotated, f"Samples: {len(collected_rows)}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow('Collect Posture Data', annotated)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break
    elif key == ord('g'):
        rows = run_countdown(cap, model, label=0, seconds=COUNTDOWN_SECS, n_frames=FRAMES_TO_CAPTURE)
        collected_rows.extend(rows)
        print(f"[GOOD]   +{len(rows)} frames | total: {len(collected_rows)}")
    elif key == ord('s'):
        rows = run_countdown(cap, model, label=1, seconds=COUNTDOWN_SECS, n_frames=FRAMES_TO_CAPTURE)
        collected_rows.extend(rows)
        print(f"[SLOUCH] +{len(rows)} frames | total: {len(collected_rows)}")

cap.release()
cv2.destroyAllWindows()
print(f"\nDone. Collected {len(collected_rows)} samples.")

Webcam open.
Press 'g' = start good posture capture | 's' = start slouch capture | 'q' = quit
[GOOD]   +30 frames | total: 30
[SLOUCH] +30 frames | total: 60
[GOOD]   +30 frames | total: 90
[SLOUCH] +30 frames | total: 120
[GOOD]   +30 frames | total: 150
[GOOD]   +30 frames | total: 180
[SLOUCH] +30 frames | total: 210
[SLOUCH] +30 frames | total: 240
[GOOD]   +30 frames | total: 270
[SLOUCH] +30 frames | total: 300

Done. Collected 300 samples.


In [8]:
df_new = pd.DataFrame(collected_rows, columns=columns)

if os.path.exists(CSV_PATH):
    # Append to existing file without writing the header again
    df_new.to_csv(CSV_PATH, mode='a', header=False, index=False)
    print(f"Appended {len(df_new)} rows to existing {CSV_PATH}")
else:
    df_new.to_csv(CSV_PATH, index=False)
    print(f"Created {CSV_PATH} with {len(df_new)} rows")

# Quick summary
total = pd.read_csv(CSV_PATH)
print(f"\nTotal in CSV: {len(total)} rows")
print(total['label'].value_counts().rename({0: 'good', 1: 'slouch'}))

Created data/posture_data.csv with 300 rows

Total in CSV: 300 rows
label
good      150
slouch    150
Name: count, dtype: int64
